# Blind search from random quantum targets

This notebook searches for Bell inequalities that separate **quantum behaviours** from the corrected tripartite cycle/siglocal polytope.

For every independently generated three-qubit behaviour $q$, it solves both LPs used in the project:

1. **Fixed-$\beta$ LP**
   $$
   \min_c c\!\cdot q,
   \qquad c\!\cdot v\ge 1\quad\forall v\in\mathrm{vert}(P_{\rm cycle}),
   \qquad c\ge0.
   $$

2. **Free-$\beta$ LP**
   $$
   \min_{c,\beta} c\!\cdot q-\beta,
   \qquad c\!\cdot v\ge\beta\quad\forall v\in\mathrm{vert}(P_{\rm cycle}),
   \qquad c\!\cdot u-\beta=1,
   $$
   where $u(abc|xyz)=1/8$ is white noise.

The code contains **no stored candidate inequality and no stored quantum realization** from earlier work. Candidates are discovered only from the seeded random samples. For each reconstructed inequality, validity on all 8,128 cycle vertices, its exact classical bound, saturation set, and facet status are audited independently of the solver's floating-point objective.

## How to use the notebook

Run the cells in order with the Julia kernel. The first code cell installs only missing Julia packages. The default is a reproducible pilot of 100 structured samples. The more general ensemble is implemented but disabled initially; enable it in the configuration cell after the pilot if desired.

The search is blind and does **not** stop when it finds a familiar or promising inequality. It completes the configured batch. Only after the batch is complete should the ranking be used to choose inequalities for NPA and dedicated see-saws.

Important conventions:

- inputs and outputs are binary;
- the 64-vector order is fixed explicitly below;
- all quantum samples use one qubit per party and binary rank-one projective measurements;
- both facets and nonfacets are retained;
- a numerical target violation is not presented as a final quantum certificate: the saved state and measurements make it reproducible, while high-precision reevaluation/NPA/see-saw belongs to the next stage.

## 1. Install the Julia dependencies if needed

`LinearAlgebra`, `Random`, `Printf`, `Dates`, `Serialization`, and `SHA` are Julia standard libraries. The only external packages are `JuMP`, `MathOptInterface`, and `GLPK`.

In [4]:
import Pkg

required_packages = ["JuMP", "MathOptInterface", "GLPK"]
missing_packages = filter(
    package -> Base.find_package(package) === nothing,
    required_packages,
)

if !isempty(missing_packages)
    println("Installing missing packages: ", join(missing_packages, ", "))
    Pkg.add(missing_packages)
else
    println("All required external Julia packages are already installed.")
end

All required external Julia packages are already installed.


## 2. Imports, coordinate order, and user configuration

The probability index is

$$
j(a,b,c,x,y,z)=1+c+2b+4a+8z+16y+32x.
$$

Thus rows of an $8\times8$ display matrix are ordered by `xyz = 000,001,...,111`, and columns by `abc = 000,001,...,111`.

The default first experiment samples 100 states in the GHZ subspace, fixes input 0 to the computational basis, and chooses the three input-1 measurements independently. Set `N_GENERAL_SAMPLES` to a positive integer to add fully Haar-random three-qubit states with all six measurements random.

In [6]:
using LinearAlgebra
using Random
using Printf
using Dates
using Serialization
using SHA
using JuMP
using GLPK

# Import MathOptInterface directly. This avoids the JuMP.MathOptInterface
# namespace error encountered with the user's Julia 1.12 installation.
import MathOptInterface as MOI

const TRIPLES = [(u, v, w) for u in 0:1 for v in 0:1 for w in 0:1]

prob_index(a, b, c, x, y, z) =
    1 + c + 2*b + 4*a + 8*z + 16*y + 32*x

state_index(a, b, c) = 1 + c + 2*b + 4*a

@assert [prob_index(a, b, c, 0, 0, 0) for (a, b, c) in TRIPLES] == collect(1:8)
@assert [prob_index(0, 0, 0, x, y, z) for (x, y, z) in TRIPLES] == collect(1:8:57)
@assert [state_index(a, b, c) for (a, b, c) in TRIPLES] == collect(1:8)

# -------------------------- USER CONFIGURATION --------------------------

# The proposed first blind batch. For a short installation test, temporarily
# set this to 2 or 3. Restore it before the research run.
N_STRUCTURED_SAMPLES = 0

# A second, broader ensemble is fully implemented. Leave it at zero for the
# first structured pilot; later set it, for example, to 100 and use a new
# output directory (or start a deliberately configured second run).
N_GENERAL_SAMPLES = 10_000

# This is a fresh, explicitly recorded seed. Every sample also receives its
# own deterministic seed, so a resumed run regenerates exactly the same target.
# Separate seed range for the general ensemble
MASTER_SEED = 31001101

SEARCH_OUTPUT_DIRECTORY =
    raw"C:\ResearchRuns\cycle_random_quantum_target_results_general_10000"

RESUME_SEARCH = true
STOP_ON_ERROR = true
WRITE_TABLES_EVERY = 500

# Reconstruction and validation tolerances. The classical validity tests
# themselves are exact integer/rational calculations after reconstruction.
RATIONAL_RECONSTRUCTION_TOLERANCE = 1e-8
LP_CONSISTENCY_TOLERANCE = 2e-7
QUANTUM_AUDIT_TOLERANCE = 2e-11
NORMALIZED_VIOLATION_TOLERANCE = 1e-9

# ------------------------------------------------------------------------

@assert N_STRUCTURED_SAMPLES >= 0
@assert N_GENERAL_SAMPLES >= 0
@assert N_STRUCTURED_SAMPLES + N_GENERAL_SAMPLES > 0
@assert WRITE_TABLES_EVERY >= 1

## 3. Generate the corrected cycle/siglocal polytope

Each deterministic strategy receives the two inputs available along one orientation of the communication cycle.

- clockwise: $a=f_A(x,z),\ b=f_B(y,x),\ c=f_C(z,y)$;
- counterclockwise: $a=f_A(x,y),\ b=f_B(y,z),\ c=f_C(z,x)$.

There are 4,096 strategies of each orientation. Their intersection consists of the 64 local deterministic behaviours, so the union has $4096+4096-64=8128$ distinct vertices.

In [8]:
"""Evaluate one of the 16 deterministic binary functions of two bits."""
function response(mask::Int, u::Int, v::Int)
    @assert 0 <= mask <= 15
    @assert u in 0:1 && v in 0:1
    return Int((mask >> (2*u + v)) & 1)
end

"""Construct one deterministic 64-component cycle behaviour."""
function deterministic_vertex(
    fA::Int,
    fB::Int,
    fC::Int,
    orientation::Symbol,
)
    vertex = zeros(Int8, 64)

    for (x, y, z) in TRIPLES
        if orientation === :clockwise
            a = response(fA, x, z)
            b = response(fB, y, x)
            c = response(fC, z, y)
        elseif orientation === :counterclockwise
            a = response(fA, x, y)
            b = response(fB, y, z)
            c = response(fC, z, x)
        else
            error("Unknown orientation: $orientation")
        end

        vertex[prob_index(a, b, c, x, y, z)] = 1
    end

    return vertex
end

"""Generate the 4096 deterministic vertices of one orientation."""
function generate_orientation(orientation::Symbol)
    vertices = Matrix{Int8}(undef, 16^3, 64)
    row = 1

    for fA in 0:15, fB in 0:15, fC in 0:15
        vertices[row, :] .= deterministic_vertex(fA, fB, fC, orientation)
        row += 1
    end

    return vertices
end

row_set(vertices) = Set(Tuple(row) for row in eachrow(vertices))

function validate_deterministic_vertices(vertices)
    @assert size(vertices, 2) == 64
    for vertex in eachrow(vertices)
        @assert all(value -> value == 0 || value == 1, vertex)
        for context in 0:7
            block = (8*context + 1):(8*context + 8)
            @assert sum(vertex[block]) == 1
        end
    end
    return true
end

validate_deterministic_vertices (generic function with 1 method)

In [9]:
vertices_clockwise = generate_orientation(:clockwise)
vertices_counterclockwise = generate_orientation(:counterclockwise)

# The 64 local deterministic behaviours occur in both orientations. Removing
# duplicates does not change the convex hull, but it exposes the true count.
vertices_cycle = unique(
    vcat(vertices_clockwise, vertices_counterclockwise);
    dims=1,
)

clockwise_set = row_set(vertices_clockwise)
counterclockwise_set = row_set(vertices_counterclockwise)

validate_deterministic_vertices(vertices_clockwise)
validate_deterministic_vertices(vertices_counterclockwise)
validate_deterministic_vertices(vertices_cycle)

@assert size(vertices_clockwise) == (4096, 64)
@assert size(vertices_counterclockwise) == (4096, 64)
@assert length(intersect(clockwise_set, counterclockwise_set)) == 64
@assert size(vertices_cycle) == (8128, 64)

const UNIFORM_BEHAVIOUR = fill(1//8, 64)

# Every probability coordinate occurs in exactly 1016 vertices. Consequently,
# the centroid of the 8128 distinct vertices is exactly white noise.
@assert all(sum(vertices_cycle; dims=1) .== 1016)
@assert [
    sum(Int(vertices_cycle[i, j]) for i in axes(vertices_cycle, 1)) //
    size(vertices_cycle, 1)
    for j in axes(vertices_cycle, 2)
] == UNIFORM_BEHAVIOUR

println("Corrected cycle polytope")
println("  clockwise vertices:        ", size(vertices_clockwise, 1))
println("  counterclockwise vertices: ", size(vertices_counterclockwise, 1))
println("  intersection:              ", length(intersect(clockwise_set, counterclockwise_set)))
println("  distinct union:            ", size(vertices_cycle, 1))

Corrected cycle polytope
  clockwise vertices:        4096
  counterclockwise vertices: 4096
  intersection:              64
  distinct union:            8128


## 4. Exact dimension and facet certification

Facet status is decided in the 50-dimensional affine hull of the polytope. The routines below compute ranks modulo several large primes and use a Hadamard bound when needed; they do not infer facet status from a floating-point rank alone.

In [11]:
# Pairwise-distinct primes fitting in Int64. Int128 is used for products.
const RANK_PRIMES = Int64[
    1_000_000_007,
    1_000_000_009,
    998_244_353,
    1_004_535_809,
]

mod_multiply(a::Int64, b::Int64, modulus::Int64) =
    Int64(mod(Int128(a) * Int128(b), Int128(modulus)))

function mod_power(base::Int64, exponent::Int64, modulus::Int64)
    result = Int64(1)
    factor = mod(base, modulus)
    power = exponent
    while power > 0
        if isodd(power)
            result = mod_multiply(result, factor, modulus)
        end
        factor = mod_multiply(factor, factor, modulus)
        power >>= 1
    end
    return result
end

"""Rank of an integer matrix over the prime field Z/pZ."""
function rank_mod_prime(
    matrix::AbstractMatrix{<:Integer},
    modulus::Int64,
)
    n_rows, n_columns = size(matrix)
    work = Matrix{Int64}(undef, n_rows, n_columns)

    @inbounds for index in eachindex(matrix)
        work[index] = mod(Int64(matrix[index]), modulus)
    end

    pivot_row = 1
    @inbounds for column in 1:n_columns
        pivot_row > n_rows && break

        pivot = 0
        for row in pivot_row:n_rows
            if work[row, column] != 0
                pivot = row
                break
            end
        end
        pivot == 0 && continue

        if pivot != pivot_row
            for j in 1:n_columns
                work[pivot_row, j], work[pivot, j] =
                    work[pivot, j], work[pivot_row, j]
            end
        end

        inverse_pivot = mod_power(work[pivot_row, column], modulus - 2, modulus)

        for row in (pivot_row + 1):n_rows
            entry = work[row, column]
            entry == 0 && continue
            factor = mod_multiply(entry, inverse_pivot, modulus)
            for j in column:n_columns
                product = mod_multiply(factor, work[pivot_row, j], modulus)
                work[row, j] = Int64(mod(
                    Int128(work[row, j]) - Int128(product),
                    Int128(modulus),
                ))
            end
        end
        pivot_row += 1
    end

    return pivot_row - 1
end

"""Certify the rational rank of an integer matrix."""
function certified_integer_rank(matrix::AbstractMatrix{<:Integer})
    n_rows, n_columns = size(matrix)
    min(n_rows, n_columns) == 0 && return 0

    ranks = [rank_mod_prime(matrix, prime) for prime in RANK_PRIMES]
    certified_rank = maximum(ranks)
    certified_rank == min(n_rows, n_columns) && return certified_rank

    maximum_squared_row_norm = maximum(
        sum(BigInt(matrix[row, column])^2 for column in 1:n_columns)
        for row in 1:n_rows
    )
    maximum_squared_row_norm == 0 && return 0

    next_minor_size = certified_rank + 1
    modulus_product = prod(BigInt(prime) for prime in RANK_PRIMES)
    hadamard_squared_bound = maximum_squared_row_norm^next_minor_size

    modulus_product^2 > hadamard_squared_bound || error(
        "The modular primes are insufficient to certify this rank exactly.",
    )
    return certified_rank
end

function difference_matrix(points::AbstractMatrix{<:Integer})
    n_points, dimension = size(points)
    n_points <= 1 && return zeros(Int64, 0, dimension)
    differences = Matrix{Int64}(undef, n_points - 1, dimension)
    @inbounds for row in 2:n_points, column in 1:dimension
        differences[row - 1, column] =
            Int64(points[row, column]) - Int64(points[1, column])
    end
    return differences
end

function certified_affine_dimension(points::AbstractMatrix{<:Integer})
    n_points = size(points, 1)
    n_points == 0 && return -1
    n_points == 1 && return 0
    return certified_integer_rank(difference_matrix(points))
end

const SIGLOCAL_AFFINE_DIMENSION = certified_affine_dimension(vertices_cycle)
@assert SIGLOCAL_AFFINE_DIMENSION == 50

println("Exact affine dimension: ", SIGLOCAL_AFFINE_DIMENSION)

Exact affine dimension: 50


## 5. Random quantum realizations

Two ensembles are available.

### Structured GHZ-subspace ensemble

- draw a Haar-random normalized vector in $\mathrm{span}\{|000\rangle,|111\rangle\}$;
- use the computational measurement for input 0 at all parties;
- draw an independent Haar-random rank-one projective measurement for input 1 at each party.

### General ensemble

- draw a Haar-random pure state in $\mathbb C^8$;
- draw all six binary rank-one projective qubit measurements independently.

No equality between parties is imposed. For each state and collection of projectors, the 64 probabilities are computed directly from the Born rule.

In [13]:
const I2 = Matrix{ComplexF64}(I, 2, 2)
const KET_ZERO = ComplexF64[1, 0]
const KET_ONE = ComplexF64[0, 1]
const COMPUTATIONAL_MEASUREMENT = [
    KET_ZERO * KET_ZERO',
    KET_ONE * KET_ONE',
]

"""Haar-random unit vector in C^dimension."""
function haar_vector(rng::AbstractRNG, dimension::Int)
    vector = randn(rng, dimension) .+ im .* randn(rng, dimension)
    return ComplexF64.(vector ./ norm(vector))
end

"""Haar-random binary rank-one projective qubit measurement."""
function random_projective_measurement(rng::AbstractRNG)
    ket = haar_vector(rng, 2)
    projector_zero = ket * ket'
    projector_one = I2 - projector_zero
    return [projector_zero, projector_one]
end

"""
Generate one structured realization. Measurements are stored as
measurements[party][input+1][output+1].
"""
function structured_realization(rng::AbstractRNG)
    amplitudes = haar_vector(rng, 2)
    state = zeros(ComplexF64, 8)
    state[state_index(0, 0, 0)] = amplitudes[1]
    state[state_index(1, 1, 1)] = amplitudes[2]

    measurements = [
        [deepcopy(COMPUTATIONAL_MEASUREMENT), random_projective_measurement(rng)]
        for _ in 1:3
    ]
    return state, measurements
end

"""Generate one fully general three-qubit realization."""
function general_realization(rng::AbstractRNG)
    state = haar_vector(rng, 8)
    measurements = [
        [random_projective_measurement(rng) for _ in 0:1]
        for _ in 1:3
    ]
    return state, measurements
end

"""Compute p(abc|xyz) from the Born rule in the canonical order."""
function quantum_behavior(state, measurements)
    @assert length(state) == 8
    probabilities = zeros(Float64, 64)
    maximum_imaginary_part = 0.0

    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        operator = kron(
            kron(measurements[1][x + 1][a + 1], measurements[2][y + 1][b + 1]),
            measurements[3][z + 1][c + 1],
        )
        expectation = dot(state, operator * state)
        maximum_imaginary_part = max(
            maximum_imaginary_part,
            abs(imag(expectation)),
        )
        probability = real(expectation)
        probabilities[prob_index(a, b, c, x, y, z)] =
            abs(probability) <= 1e-14 ? 0.0 : probability
    end

    return probabilities, maximum_imaginary_part
end

quantum_behavior

## 6. Audit every generated quantum behaviour

Before either LP is solved, the notebook checks the state norm, projector properties, measurement completeness, Born probabilities, normalization, and all two-party nonsignalling marginals. A failed audit stops the default run and writes a failure record.

In [15]:
function nonsignalling_residuals(probabilities)
    normalization = 0.0
    ab = 0.0
    ac = 0.0
    bc = 0.0

    for (x, y, z) in TRIPLES
        block_sum = sum(
            probabilities[prob_index(a, b, c, x, y, z)]
            for (a, b, c) in TRIPLES
        )
        normalization = max(normalization, abs(block_sum - 1))
    end

    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, y, 0)] for c in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, y, 1)] for c in 0:1)
        ab = max(ab, abs(lhs - rhs))
    end

    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, 0, z)] for b in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, 1, z)] for b in 0:1)
        ac = max(ac, abs(lhs - rhs))
    end

    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, 0, y, z)] for a in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, 1, y, z)] for a in 0:1)
        bc = max(bc, abs(lhs - rhs))
    end

    return (normalization=normalization, ab=ab, ac=ac, bc=bc)
end

"""Return diagnostics and throw if a realization is not a valid quantum model."""
function audit_quantum_realization(
    state,
    measurements,
    probabilities,
    maximum_imaginary_part;
    tolerance=QUANTUM_AUDIT_TOLERANCE,
)
    state_norm_error = abs(real(dot(state, state)) - 1)
    maximum_hermiticity_error = 0.0
    maximum_idempotence_error = 0.0
    maximum_completeness_error = 0.0
    minimum_projector_eigenvalue = Inf

    for party in 1:3, input in 1:2
        projectors = measurements[party][input]
        @assert length(projectors) == 2
        maximum_completeness_error = max(
            maximum_completeness_error,
            opnorm(projectors[1] + projectors[2] - I2),
        )

        for projector in projectors
            maximum_hermiticity_error = max(
                maximum_hermiticity_error,
                opnorm(projector - projector'),
            )
            maximum_idempotence_error = max(
                maximum_idempotence_error,
                opnorm(projector * projector - projector),
            )
            minimum_projector_eigenvalue = min(
                minimum_projector_eigenvalue,
                minimum(eigvals(Hermitian(projector))),
            )
        end
    end

    ns = nonsignalling_residuals(probabilities)
    minimum_probability = minimum(probabilities)
    maximum_probability = maximum(probabilities)

    state_norm_error <= tolerance || error("State normalization audit failed")
    maximum_hermiticity_error <= tolerance || error("Hermiticity audit failed")
    maximum_idempotence_error <= tolerance || error("Projectivity audit failed")
    maximum_completeness_error <= tolerance || error("Measurement completeness audit failed")
    minimum_projector_eigenvalue >= -tolerance || error("A projector is not positive")
    maximum_imaginary_part <= tolerance || error("Born probability has a non-negligible imaginary part")
    minimum_probability >= -tolerance || error("A Born probability is negative")
    maximum_probability <= 1 + tolerance || error("A Born probability exceeds one")
    ns.normalization <= tolerance || error("Probability normalization audit failed")
    maximum((ns.ab, ns.ac, ns.bc)) <= tolerance || error("Nonsignalling audit failed")

    return Dict{String, Any}(
        "state_norm_error" => state_norm_error,
        "measurement_hermiticity_error" => maximum_hermiticity_error,
        "measurement_idempotence_error" => maximum_idempotence_error,
        "measurement_completeness_error" => maximum_completeness_error,
        "minimum_projector_eigenvalue" => minimum_projector_eigenvalue,
        "born_imaginary_error" => maximum_imaginary_part,
        "minimum_probability" => minimum_probability,
        "maximum_probability" => maximum_probability,
        "normalization_residual" => ns.normalization,
        "ab_nonsignalling_residual" => ns.ab,
        "ac_nonsignalling_residual" => ns.ac,
        "bc_nonsignalling_residual" => ns.bc,
    )
end

audit_quantum_realization

## 7. Build the two reusable LPs

The models contain all 8,128 corrected vertices. Only their objectives change from one quantum target to the next.

The 64-coordinate representation is retained directly. The free-$\beta$ model therefore has the usual affine-gauge freedom in its coefficient vector. Later deduplication uses the exact slack on every cycle vertex, so coefficient vectors differing only by affine equalities are recognized as the same geometric inequality.

In [17]:
function build_fixed_beta_model(vertices)
    number_of_vertices, dimension = size(vertices)
    model = Model(GLPK.Optimizer)
    set_silent(model)

    @variable(model, coefficients[1:dimension] >= 0)
    @constraint(
        model,
        valid_on_cycle[i=1:number_of_vertices],
        sum(vertices[i, j] * coefficients[j] for j in 1:dimension) >= 1,
    )
    @objective(model, Min, sum(coefficients))
    return model, coefficients
end

function build_free_beta_model(vertices, uniform_behavior)
    number_of_vertices, dimension = size(vertices)
    model = Model(GLPK.Optimizer)
    set_silent(model)

    # Variables are unrestricted unless a bound is supplied.
    @variable(model, coefficients[1:dimension])
    @variable(model, beta)
    @constraint(
        model,
        valid_on_cycle[i=1:number_of_vertices],
        sum(vertices[i, j] * coefficients[j] for j in 1:dimension) >= beta,
    )
    @constraint(
        model,
        white_noise_normalization,
        sum(Float64(uniform_behavior[j]) * coefficients[j] for j in 1:dimension) - beta == 1,
    )
    @objective(model, Min, sum(coefficients) - beta)
    return model, coefficients, beta
end

function solve_fixed_beta_target!(model, coefficients, target)
    set_objective_coefficient(model, coefficients, Float64.(target))
    optimize!(model)
    status = termination_status(model)
    status == MOI.OPTIMAL || error("Fixed-beta LP ended with status $status")
    return (
        coefficients=value.(coefficients),
        beta=1.0,
        solver_objective=objective_value(model),
        status=string(status),
    )
end

function solve_free_beta_target!(model, coefficients, beta, target)
    set_objective_coefficient(model, coefficients, Float64.(target))
    set_objective_coefficient(model, beta, -1.0)
    optimize!(model)
    status = termination_status(model)
    status == MOI.OPTIMAL || error("Free-beta LP ended with status $status")
    return (
        coefficients=value.(coefficients),
        beta=value(beta),
        solver_objective=objective_value(model),
        status=string(status),
    )
end

fixed_beta_model, fixed_beta_coefficients =
    build_fixed_beta_model(vertices_cycle)

free_beta_model, free_beta_coefficients, free_beta_beta =
    build_free_beta_model(vertices_cycle, UNIFORM_BEHAVIOUR)

println("Both reusable LP models were built successfully.")

Both reusable LP models were built successfully.


## 8. Exact reconstruction, validity, facet status, and robustness

GLPK returns floating-point coefficient vectors. For every solution the notebook:

1. rationally reconstructs all coefficients (and $\beta$ in the free LP);
2. clears denominators;
3. computes the **exact** minimum over all 8,128 vertices and uses it as the certified bound $B$;
4. requires that the reconstructed target objective agrees with the LP objective;
5. reduces $(C,B)$ to primitive integers;
6. evaluates the quantum target and white noise;
7. certifies the dimension of the saturating face.

For $C\cdot p\ge B$, define

$$
\Delta=B-C\cdot q,\qquad
W=C\cdot u-B,\qquad
G=\frac{\Delta}{W}.
$$

When $\Delta>0$, the white-noise fraction tolerated by this particular target is

$$
\eta=\frac{G}{1+G}.
$$

These quantities are invariant under positive rescaling and under affine-gauge changes on the polytope.

In [19]:
function rationalize_scalar(value; tolerance=RATIONAL_RECONSTRUCTION_TOLERANCE)
    isfinite(value) || error("Cannot rationalize a non-finite LP coefficient")
    return abs(value) <= tolerance ?
        BigInt(0)//BigInt(1) :
        rationalize(BigInt, value; tol=tolerance)
end

function rationalize_solution(coefficients_float, beta_float)
    coefficients = Rational{BigInt}[
        rationalize_scalar(value) for value in coefficients_float
    ]
    beta = rationalize_scalar(beta_float)
    return coefficients, beta
end

"""Clear denominators simultaneously from coefficients and beta."""
function integer_form(coefficients, beta)
    all_values = vcat(coefficients, [beta])
    common_denominator = foldl(
        lcm,
        denominator.(all_values);
        init=BigInt(1),
    )
    integers = BigInt[
        numerator(value * common_denominator) for value in all_values
    ]
    return integers[1:end-1], integers[end], common_denominator
end

"""Evaluate an integer coefficient vector on every deterministic vertex."""
function integer_vertex_values(vertices, coefficients)
    number_of_vertices, dimension = size(vertices)
    @assert length(coefficients) == dimension
    values = Vector{BigInt}(undef, number_of_vertices)

    @inbounds for row in 1:number_of_vertices
        total = BigInt(0)
        for column in 1:dimension
            if vertices[row, column] != 0
                total += coefficients[column]
            end
        end
        values[row] = total
    end
    return values
end

function primitive_integer_inequality(coefficients, bound)
    common_divisor = foldl(
        gcd,
        abs.(vcat(coefficients, [bound]));
        init=BigInt(0),
    )
    common_divisor == 0 && error("The reconstructed inequality is zero")
    return coefficients .÷ common_divisor, bound ÷ common_divisor
end

function relative_agreement(actual, expected; tolerance=LP_CONSISTENCY_TOLERANCE)
    return abs(actual - expected) <= tolerance * (1 + abs(expected))
end

"""
Return an exact, gauge- and scale-invariant key from all vertex slacks.
The full string is used as the dictionary key; SHA-256 is only a readable ID.
"""
function canonical_slack_key(vertex_values, bound)
    slacks = vertex_values .- bound
    minimum(slacks) == 0 || error("A certified bound must be saturated")
    minimum(slacks) >= 0 || error("The reconstructed inequality is invalid")

    nonzero_slacks = filter(value -> !iszero(value), slacks)
    isempty(nonzero_slacks) && return nothing, nothing
    divisor = foldl(gcd, abs.(nonzero_slacks); init=BigInt(0))
    canonical_slacks = slacks .÷ divisor
    key = join(string.(canonical_slacks), ",")
    identifier = bytes2hex(sha256(codeunits(key)))
    return key, identifier
end

"""Exact affine dimension of a proper face; facet detection can stop at 49."""
function certified_face_dimension(points)
    size(points, 1) == 0 && return -1
    size(points, 1) == 1 && return 0
    differences = difference_matrix(points)

    # Because the inequality is valid, tight, and nonconstant on a
    # 50-dimensional polytope, this face has dimension at most 49. Witnessing
    # rank 49 modulo one prime therefore certifies a facet immediately.
    first_rank = rank_mod_prime(differences, first(RANK_PRIMES))
    first_rank == SIGLOCAL_AFFINE_DIMENSION - 1 && return first_rank

    return certified_integer_rank(differences)
end

certified_face_dimension

In [20]:
"""Audit one numerical LP solution and return a self-contained outcome."""
function audit_lp_solution(
    lp_kind::String,
    numerical_solution,
    target,
    vertices_cycle,
    vertices_clockwise,
    vertices_counterclockwise,
)
    coefficients_rational, beta_rational = rationalize_solution(
        numerical_solution.coefficients,
        numerical_solution.beta,
    )
    coefficients_raw, beta_raw, common_denominator = integer_form(
        coefficients_rational,
        beta_rational,
    )

    if lp_kind == "fixed_beta"
        minimum(coefficients_raw) >= 0 || error(
            "Fixed-beta reconstruction produced a negative coefficient",
        )
    elseif lp_kind != "free_beta"
        error("Unknown LP kind: $lp_kind")
    end

    cycle_values_raw = integer_vertex_values(vertices_cycle, coefficients_raw)
    exact_bound_raw = minimum(cycle_values_raw)
    reconstructed_bound_float = Float64(exact_bound_raw // common_denominator)

    # Check that rational reconstruction preserves the solver's active bound.
    expected_beta = numerical_solution.beta
    relative_agreement(reconstructed_bound_float, expected_beta) || error(
        "Reconstructed classical bound disagrees with the LP beta",
    )

    reconstructed_target_linear = sum(
        Float64(coefficients_raw[j] // common_denominator) * target[j]
        for j in eachindex(target)
    )
    reconstructed_solver_objective = if lp_kind == "fixed_beta"
        reconstructed_target_linear
    else
        reconstructed_target_linear - Float64(beta_raw // common_denominator)
    end
    relative_agreement(
        reconstructed_solver_objective,
        numerical_solution.solver_objective,
    ) || error("Rational reconstruction changed the LP objective")

    coefficients, bound = primitive_integer_inequality(
        coefficients_raw,
        exact_bound_raw,
    )

    cycle_values = integer_vertex_values(vertices_cycle, coefficients)
    clockwise_values = integer_vertex_values(vertices_clockwise, coefficients)
    counterclockwise_values = integer_vertex_values(vertices_counterclockwise, coefficients)
    exact_bound = minimum(cycle_values)
    exact_bound == bound || error("Primitive reduction changed the bound")
    minimum(cycle_values .- bound) == 0 || error("The exact inequality is not tight")
    minimum(cycle_values .- bound) >= 0 || error("The exact inequality is invalid")

    uniform_slack = sum(coefficients) // 8 - bound
    target_value = sum(Float64(coefficients[j]) * target[j] for j in eachindex(target))
    gap = Float64(bound) - target_value

    # In the fixed-beta LP an interior target can return an affine equality,
    # such as normalization. It is valid but is not a Bell inequality and has
    # no white-noise normalization. Record it as a noncandidate LP outcome.
    if uniform_slack == 0
        all(cycle_values .== bound) || error(
            "Zero uniform slack is inconsistent with the vertex centroid",
        )
        return Dict{String, Any}(
            "lp_kind" => lp_kind,
            "status" => numerical_solution.status,
            "solver_objective" => numerical_solution.solver_objective,
            "outcome_kind" => "affine_equality",
            "candidate" => nothing,
            "target_value" => target_value,
            "gap" => gap,
            "normalized_gap" => NaN,
            "critical_visibility" => NaN,
            "critical_noise_fraction" => NaN,
            "target_violates" => false,
        )
    end

    uniform_slack > 0 || error("The reconstructed inequality has negative white-noise slack")
    normalized_gap = gap / Float64(uniform_slack)
    target_violates = normalized_gap > NORMALIZED_VIOLATION_TOLERANCE
    critical_visibility = target_violates ? 1 / (1 + normalized_gap) : NaN
    critical_noise_fraction = target_violates ? normalized_gap / (1 + normalized_gap) : NaN

    saturating_indices = findall(==(bound), cycle_values)
    isempty(saturating_indices) && error("The reconstructed inequality is not saturated")
    face_dimension = certified_face_dimension(vertices_cycle[saturating_indices, :])

    key, candidate_id = canonical_slack_key(cycle_values, bound)
    key === nothing && error("A nonconstant inequality has a constant slack vector")

    candidate = Dict{String, Any}(
        "candidate_id" => candidate_id,
        "canonical_slack_key" => key,
        "coefficients" => coefficients,
        "bound" => bound,
        "uniform_slack" => uniform_slack,
        "cycle_minimum" => minimum(cycle_values),
        "clockwise_minimum" => minimum(clockwise_values),
        "counterclockwise_minimum" => minimum(counterclockwise_values),
        "saturation_count" => length(saturating_indices),
        "face_dimension" => face_dimension,
        "is_facet" => face_dimension == SIGLOCAL_AFFINE_DIMENSION - 1,
    )

    return Dict{String, Any}(
        "lp_kind" => lp_kind,
        "status" => numerical_solution.status,
        "solver_objective" => numerical_solution.solver_objective,
        "outcome_kind" => "inequality",
        "candidate" => candidate,
        "target_value" => target_value,
        "gap" => gap,
        "normalized_gap" => normalized_gap,
        "critical_visibility" => critical_visibility,
        "critical_noise_fraction" => critical_noise_fraction,
        "target_violates" => target_violates,
    )
end

audit_lp_solution

## 9. Checkpoints and human-readable output files

The binary checkpoint is written atomically after every completed sample. The TSV files are rewritten atomically every few samples and at the end. They include:

- the random sample seeds and audit residuals;
- every LP result, including nonviolations and affine equalities;
- every distinct exact inequality, with facets and nonfacets retained;
- a ranking by the best target-specific white-noise tolerance found in this blind batch;
- every state, projector matrix, and probability needed to reproduce the sampled quantum behaviours;
- failures, if any.

Deduplication is exact on the complete normalized slack vector over the 8,128 vertices. It does not use rounded floating-point coefficients and does not identify symmetry-related but genuinely different slack vectors.

In [22]:
function rational_string(value::Rational)
    denominator(value) == 1 && return string(numerator(value))
    return "$(numerator(value))/$(denominator(value))"
end

float_string(value) = isfinite(Float64(value)) ? @sprintf("%.17g", Float64(value)) : ""
integer_vector_string(values) = join(string.(values), ",")
safe_text(value) = replace(string(value), '\t' => ' ', '\n' => ' ')

function atomic_write(writer, path)
    temporary_path = path * ".tmp"
    open(temporary_path, "w") do io
        writer(io)
    end
    mv(temporary_path, path; force=true)
    return path
end

function write_checkpoint(path, state)
    return atomic_write(path) do io
        serialize(io, state)
    end
end

function read_checkpoint(path)
    return open(path, "r") do io
        deserialize(io)
    end
end

function append_failure(path, sample_id, ensemble, stage, error_value)
    new_file = !isfile(path)
    open(path, "a") do io
        if new_file
            println(io, "timestamp\tsample_id\tensemble\tstage\tmessage")
        end
        println(io, join([
            string(now()),
            string(sample_id),
            string(ensemble),
            string(stage),
            safe_text(sprint(showerror, error_value)),
        ], '\t'))
    end
end

function output_paths(output_directory)
    return Dict(
        "checkpoint" => joinpath(output_directory, "search_checkpoint.jls"),
        "samples" => joinpath(output_directory, "samples.tsv"),
        "lp_runs" => joinpath(output_directory, "lp_runs.tsv"),
        "candidates" => joinpath(output_directory, "unique_candidates.tsv"),
        "ranking" => joinpath(output_directory, "candidate_ranking.tsv"),
        "states" => joinpath(output_directory, "quantum_states.tsv"),
        "measurements" => joinpath(output_directory, "quantum_measurements.tsv"),
        "behaviors" => joinpath(output_directory, "quantum_behaviors.tsv"),
        "failures" => joinpath(output_directory, "failures.tsv"),
        "metadata" => joinpath(output_directory, "run_metadata.txt"),
    )
end

output_paths (generic function with 1 method)

In [23]:
function write_sample_table(path, samples)
    headers = [
        "sample_id", "ensemble", "ensemble_index", "seed",
        "state_norm_error", "measurement_hermiticity_error",
        "measurement_idempotence_error", "measurement_completeness_error",
        "minimum_projector_eigenvalue", "born_imaginary_error",
        "minimum_probability", "maximum_probability", "normalization_residual",
        "ab_nonsignalling_residual", "ac_nonsignalling_residual",
        "bc_nonsignalling_residual",
    ]
    atomic_write(path) do io
        println(io, join(headers, '\t'))
        for sample in samples
            audit = sample["audit"]
            println(io, join([
                string(sample["sample_id"]), string(sample["ensemble"]),
                string(sample["ensemble_index"]), string(sample["seed"]),
                float_string(audit["state_norm_error"]),
                float_string(audit["measurement_hermiticity_error"]),
                float_string(audit["measurement_idempotence_error"]),
                float_string(audit["measurement_completeness_error"]),
                float_string(audit["minimum_projector_eigenvalue"]),
                float_string(audit["born_imaginary_error"]),
                float_string(audit["minimum_probability"]),
                float_string(audit["maximum_probability"]),
                float_string(audit["normalization_residual"]),
                float_string(audit["ab_nonsignalling_residual"]),
                float_string(audit["ac_nonsignalling_residual"]),
                float_string(audit["bc_nonsignalling_residual"]),
            ], '\t'))
        end
    end
end

function write_lp_run_table(path, runs)
    headers = [
        "sample_id", "ensemble", "seed", "lp_kind", "solver_status",
        "solver_objective", "outcome_kind", "candidate_id", "is_duplicate",
        "classical_bound", "target_value", "gap", "uniform_slack",
        "normalized_gap", "target_violates", "critical_visibility",
        "critical_noise_fraction", "saturation_count", "face_dimension", "is_facet",
    ]
    atomic_write(path) do io
        println(io, join(headers, '\t'))
        for run in runs
            println(io, join([
                string(run["sample_id"]), string(run["ensemble"]), string(run["seed"]),
                run["lp_kind"], run["status"], float_string(run["solver_objective"]),
                run["outcome_kind"], run["candidate_id"], string(run["is_duplicate"]),
                run["classical_bound"], float_string(run["target_value"]),
                float_string(run["gap"]), run["uniform_slack"],
                float_string(run["normalized_gap"]), string(run["target_violates"]),
                float_string(run["critical_visibility"]),
                float_string(run["critical_noise_fraction"]),
                string(run["saturation_count"]), string(run["face_dimension"]),
                string(run["is_facet"]),
            ], '\t'))
        end
    end
end

function write_candidate_table(path, candidates)
    headers = [
        "candidate_id", "first_sample_id", "best_sample_id", "lp_kinds",
        "discovery_count", "bound", "uniform_slack", "cycle_minimum",
        "clockwise_minimum", "counterclockwise_minimum", "saturation_count",
        "face_dimension", "is_facet", "best_target_value", "best_gap",
        "best_normalized_gap", "best_critical_visibility",
        "best_critical_noise_fraction", "coefficients_in_canonical_64_order",
    ]
    atomic_write(path) do io
        println(io, join(headers, '\t'))
        for candidate in candidates
            println(io, join([
                candidate["candidate_id"], string(candidate["first_sample_id"]),
                string(candidate["best_sample_id"]), join(candidate["lp_kinds"], ","),
                string(candidate["discovery_count"]), string(candidate["bound"]),
                rational_string(candidate["uniform_slack"]),
                string(candidate["cycle_minimum"]), string(candidate["clockwise_minimum"]),
                string(candidate["counterclockwise_minimum"]),
                string(candidate["saturation_count"]), string(candidate["face_dimension"]),
                string(candidate["is_facet"]), float_string(candidate["best_target_value"]),
                float_string(candidate["best_gap"]), float_string(candidate["best_normalized_gap"]),
                float_string(candidate["best_critical_visibility"]),
                float_string(candidate["best_critical_noise_fraction"]),
                integer_vector_string(candidate["coefficients"]),
            ], '\t'))
        end
    end
end

function write_ranking_table(path, candidates)
    violating = filter(candidate -> candidate["best_normalized_gap"] > NORMALIZED_VIOLATION_TOLERANCE, candidates)
    ranking = sort(violating; by=candidate -> candidate["best_critical_noise_fraction"], rev=true)

    atomic_write(path) do io
        println(io, join([
            "rank", "candidate_id", "is_facet", "face_dimension", "lp_kinds",
            "best_sample_id", "bound", "best_target_value", "best_normalized_gap",
            "best_critical_visibility", "best_critical_noise_fraction",
        ], '\t'))
        for (rank_index, candidate) in enumerate(ranking)
            println(io, join([
                string(rank_index), candidate["candidate_id"], string(candidate["is_facet"]),
                string(candidate["face_dimension"]), join(candidate["lp_kinds"], ","),
                string(candidate["best_sample_id"]), string(candidate["bound"]),
                float_string(candidate["best_target_value"]),
                float_string(candidate["best_normalized_gap"]),
                float_string(candidate["best_critical_visibility"]),
                float_string(candidate["best_critical_noise_fraction"]),
            ], '\t'))
        end
    end
end

write_ranking_table (generic function with 1 method)

In [24]:
function write_realization_tables(paths, samples)
    atomic_write(paths["states"]) do io
        println(io, "sample_id\tbasis_index\ta\tb\tc\treal\timag")
        for sample in samples, (a, b, c) in TRIPLES
            index = state_index(a, b, c)
            amplitude = sample["state"][index]
            println(io, join([
                string(sample["sample_id"]), string(index), string(a), string(b), string(c),
                float_string(real(amplitude)), float_string(imag(amplitude)),
            ], '\t'))
        end
    end

    atomic_write(paths["measurements"]) do io
        println(io, "sample_id\tparty\tinput\toutput\trow\tcolumn\treal\timag")
        for sample in samples, party in 1:3, input in 0:1, output in 0:1, row in 1:2, column in 1:2
            entry = sample["measurements"][party][input + 1][output + 1][row, column]
            println(io, join([
                string(sample["sample_id"]), string(party), string(input), string(output),
                string(row), string(column), float_string(real(entry)), float_string(imag(entry)),
            ], '\t'))
        end
    end

    atomic_write(paths["behaviors"]) do io
        println(io, "sample_id\tx\ty\tz\ta\tb\tc\tprobability")
        for sample in samples, (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
            probability = sample["behavior"][prob_index(a, b, c, x, y, z)]
            println(io, join([
                string(sample["sample_id"]), string(x), string(y), string(z),
                string(a), string(b), string(c), float_string(probability),
            ], '\t'))
        end
    end
end

function write_metadata(path, state, vertex_fingerprint)
    config = state["config"]
    atomic_write(path) do io
        println(io, "Blind random-quantum-target search")
        println(io, "saved_at = ", now())
        println(io, "julia_version = ", VERSION)
        println(io, "format_version = ", state["format_version"])
        println(io, "master_seed = ", config["master_seed"])
        println(io, "structured_samples = ", config["n_structured"])
        println(io, "general_samples = ", config["n_general"])
        println(io, "completed_samples = ", length(state["samples"]))
        println(io, "lp_runs = ", length(state["lp_runs"]))
        println(io, "unique_candidates = ", length(state["candidates"]))
        println(io, "failures = ", length(state["failures"]))
        println(io, "vertex_count = ", size(vertices_cycle, 1))
        println(io, "vertex_fingerprint_sha256 = ", vertex_fingerprint)
        println(io, "coordinate_order = 1+c+2b+4a+8z+16y+32x")
        println(io, "fixed_beta_lp = min c.q; c.v>=1; c>=0")
        println(io, "free_beta_lp = min c.q-beta; c.v>=beta; c.u-beta=1; c,beta free")
        println(io, "rational_reconstruction_tolerance = ", config["rational_tolerance"])
        println(io, "lp_consistency_tolerance = ", config["lp_consistency_tolerance"])
        println(io, "quantum_audit_tolerance = ", config["quantum_audit_tolerance"])
        println(io, "normalized_violation_tolerance = ", config["normalized_violation_tolerance"])
    end
end

function write_all_tables!(paths, state, vertex_fingerprint)
    write_sample_table(paths["samples"], state["samples"])
    write_lp_run_table(paths["lp_runs"], state["lp_runs"])
    write_candidate_table(paths["candidates"], state["candidates"])
    write_ranking_table(paths["ranking"], state["candidates"])
    write_realization_tables(paths, state["samples"])
    write_metadata(paths["metadata"], state, vertex_fingerprint)
    return nothing
end

write_all_tables! (generic function with 1 method)

## 10. Search engine

Each sample has an independent seed. The sample is processed as a transaction:

1. generate and audit the realization;
2. solve and audit the fixed-$\beta$ LP;
3. solve and audit the free-$\beta$ LP;
4. only then append the sample and both results to the persistent state.

If anything fails before step 4, the default fail-fast run records the error and leaves that sample as the next one to retry after correction. This avoids incomplete samples in the result tables.

In [26]:
function vertex_fingerprint(vertices)
    bytes = reinterpret(UInt8, vec(copy(vertices)))
    return bytes2hex(sha256(bytes))
end

const VERTEX_FINGERPRINT = vertex_fingerprint(vertices_cycle)

function search_config()
    return Dict{String, Any}(
        "master_seed" => MASTER_SEED,
        "n_structured" => N_STRUCTURED_SAMPLES,
        "n_general" => N_GENERAL_SAMPLES,
        "rational_tolerance" => RATIONAL_RECONSTRUCTION_TOLERANCE,
        "lp_consistency_tolerance" => LP_CONSISTENCY_TOLERANCE,
        "quantum_audit_tolerance" => QUANTUM_AUDIT_TOLERANCE,
        "normalized_violation_tolerance" => NORMALIZED_VIOLATION_TOLERANCE,
        "vertex_fingerprint" => VERTEX_FINGERPRINT,
        "coordinate_order" => "1+c+2b+4a+8z+16y+32x",
    )
end

function fresh_state()
    return Dict{String, Any}(
        "format_version" => 1,
        "config" => search_config(),
        "next_sample_id" => 1,
        "samples" => Any[],
        "lp_runs" => Any[],
        "candidates" => Any[],
        "candidate_index" => Dict{String, Int}(),
        "failures" => Any[],
    )
end

function validate_checkpoint(state)
    state["format_version"] == 1 || error("Checkpoint format mismatch")
    state["config"] == search_config() || error(
        "Checkpoint configuration mismatch. Use the same configuration or a new output directory.",
    )
    all(
        sample["sample_id"] < state["next_sample_id"]
        for sample in state["samples"]
    ) || error("Checkpoint sample index is inconsistent")
    length(state["lp_runs"]) == 2 * length(state["samples"]) || error(
        "Checkpoint does not contain exactly two LP results per committed sample",
    )
    return true
end

function sample_spec(sample_id)
    total_samples = N_STRUCTURED_SAMPLES + N_GENERAL_SAMPLES
    1 <= sample_id <= total_samples || error("Sample ID is outside the configured plan")

    if sample_id <= N_STRUCTURED_SAMPLES
        ensemble = "structured_ghz"
        ensemble_index = sample_id
    else
        ensemble = "haar_general"
        ensemble_index = sample_id - N_STRUCTURED_SAMPLES
    end

    seed = MASTER_SEED + sample_id - 1
    return ensemble, ensemble_index, seed
end

function generate_sample(sample_id)
    ensemble, ensemble_index, seed = sample_spec(sample_id)
    rng = MersenneTwister(seed)
    state, measurements = if ensemble == "structured_ghz"
        structured_realization(rng)
    else
        general_realization(rng)
    end

    behavior, imaginary_error = quantum_behavior(state, measurements)
    audit = audit_quantum_realization(
        state,
        measurements,
        behavior,
        imaginary_error,
    )

    return Dict{String, Any}(
        "sample_id" => sample_id,
        "ensemble" => ensemble,
        "ensemble_index" => ensemble_index,
        "seed" => seed,
        "state" => state,
        "measurements" => measurements,
        "behavior" => behavior,
        "audit" => audit,
    )
end

function solve_both_lps(sample)
    target = sample["behavior"]

    fixed_numerical = solve_fixed_beta_target!(
        fixed_beta_model,
        fixed_beta_coefficients,
        target,
    )
    fixed_outcome = audit_lp_solution(
        "fixed_beta",
        fixed_numerical,
        target,
        vertices_cycle,
        vertices_clockwise,
        vertices_counterclockwise,
    )

    free_numerical = solve_free_beta_target!(
        free_beta_model,
        free_beta_coefficients,
        free_beta_beta,
        target,
    )
    free_outcome = audit_lp_solution(
        "free_beta",
        free_numerical,
        target,
        vertices_cycle,
        vertices_clockwise,
        vertices_counterclockwise,
    )

    return [fixed_outcome, free_outcome]
end

solve_both_lps (generic function with 1 method)

In [27]:
function target_metrics_for_representative(candidate, target)
    target_value = sum(
        Float64(candidate["coefficients"][j]) * target[j]
        for j in eachindex(target)
    )
    gap = Float64(candidate["bound"]) - target_value
    normalized_gap = gap / Float64(candidate["uniform_slack"])
    target_violates = normalized_gap > NORMALIZED_VIOLATION_TOLERANCE
    critical_visibility = target_violates ? 1 / (1 + normalized_gap) : NaN
    critical_noise_fraction = target_violates ? normalized_gap / (1 + normalized_gap) : NaN
    return (
        target_value=target_value,
        gap=gap,
        normalized_gap=normalized_gap,
        critical_visibility=critical_visibility,
        critical_noise_fraction=critical_noise_fraction,
    )
end

function update_unique_candidate!(state, sample, outcome)
    candidate = outcome["candidate"]
    candidate === nothing && return "", false

    key = candidate["canonical_slack_key"]
    index = get(state["candidate_index"], key, 0)
    is_duplicate = index != 0

    if !is_duplicate
        stored = deepcopy(candidate)
        metrics = target_metrics_for_representative(stored, sample["behavior"])
        stored["first_sample_id"] = sample["sample_id"]
        stored["best_sample_id"] = sample["sample_id"]
        stored["lp_kinds"] = String[outcome["lp_kind"]]
        stored["discovery_count"] = 1
        stored["best_target_value"] = metrics.target_value
        stored["best_gap"] = metrics.gap
        stored["best_normalized_gap"] = metrics.normalized_gap
        stored["best_critical_visibility"] = metrics.critical_visibility
        stored["best_critical_noise_fraction"] = metrics.critical_noise_fraction
        push!(state["candidates"], stored)
        index = length(state["candidates"])
        state["candidate_index"][key] = index
    else
        stored = state["candidates"][index]
        metrics = target_metrics_for_representative(stored, sample["behavior"])
        relative_agreement(metrics.normalized_gap, outcome["normalized_gap"]) || error(
            "Gauge-equivalent representatives disagree on normalized target slack",
        )
        stored["discovery_count"] += 1
        if !(outcome["lp_kind"] in stored["lp_kinds"])
            push!(stored["lp_kinds"], outcome["lp_kind"])
            sort!(stored["lp_kinds"])
        end
        if metrics.normalized_gap > stored["best_normalized_gap"]
            stored["best_sample_id"] = sample["sample_id"]
            stored["best_target_value"] = metrics.target_value
            stored["best_gap"] = metrics.gap
            stored["best_normalized_gap"] = metrics.normalized_gap
            stored["best_critical_visibility"] = metrics.critical_visibility
            stored["best_critical_noise_fraction"] = metrics.critical_noise_fraction
        end
    end

    return state["candidates"][index]["candidate_id"], is_duplicate
end

function make_lp_run_record(sample, outcome, candidate_id, is_duplicate)
    candidate = outcome["candidate"]
    return Dict{String, Any}(
        "sample_id" => sample["sample_id"],
        "ensemble" => sample["ensemble"],
        "seed" => sample["seed"],
        "lp_kind" => outcome["lp_kind"],
        "status" => outcome["status"],
        "solver_objective" => outcome["solver_objective"],
        "outcome_kind" => outcome["outcome_kind"],
        "candidate_id" => candidate_id,
        "is_duplicate" => is_duplicate,
        "classical_bound" => candidate === nothing ? "" : string(candidate["bound"]),
        "target_value" => outcome["target_value"],
        "gap" => outcome["gap"],
        "uniform_slack" => candidate === nothing ? "" : rational_string(candidate["uniform_slack"]),
        "normalized_gap" => outcome["normalized_gap"],
        "target_violates" => outcome["target_violates"],
        "critical_visibility" => outcome["critical_visibility"],
        "critical_noise_fraction" => outcome["critical_noise_fraction"],
        "saturation_count" => candidate === nothing ? 0 : candidate["saturation_count"],
        "face_dimension" => candidate === nothing ? -1 : candidate["face_dimension"],
        "is_facet" => candidate === nothing ? false : candidate["is_facet"],
    )
end

"""Commit one completely processed sample and its two LP outcomes."""
function commit_sample!(state, sample, outcomes)
    @assert length(outcomes) == 2
    @assert Set(outcome["lp_kind"] for outcome in outcomes) == Set(["fixed_beta", "free_beta"])

    push!(state["samples"], sample)
    for outcome in outcomes
        candidate_id, is_duplicate = update_unique_candidate!(state, sample, outcome)
        push!(
            state["lp_runs"],
            make_lp_run_record(sample, outcome, candidate_id, is_duplicate),
        )
    end
    state["next_sample_id"] = sample["sample_id"] + 1
    return nothing
end

commit_sample!

In [28]:
function run_blind_search!(;
    output_directory=SEARCH_OUTPUT_DIRECTORY,
    resume=RESUME_SEARCH,
    stop_on_error=STOP_ON_ERROR,
    write_tables_every=WRITE_TABLES_EVERY,
)
    mkpath(output_directory)
    paths = output_paths(output_directory)

    state = if resume && isfile(paths["checkpoint"])
        loaded = read_checkpoint(paths["checkpoint"])
        validate_checkpoint(loaded)
        println(
            "Resuming at sample ", loaded["next_sample_id"],
            " after ", length(loaded["samples"]), " completed samples.",
        )
        loaded
    else
        if !resume && any(isfile(path) for path in values(paths))
            error(
                "RESUME_SEARCH=false but the output directory contains result files. " *
                "Choose a new output directory to avoid overwriting a research run.",
            )
        end
        fresh_state()
    end

    total_samples = N_STRUCTURED_SAMPLES + N_GENERAL_SAMPLES
    started_at = time()

    try
        while state["next_sample_id"] <= total_samples
            sample_id = state["next_sample_id"]
            ensemble, ensemble_index, seed = sample_spec(sample_id)
            println(
                "\nSample $sample_id/$total_samples | $ensemble #$ensemble_index | seed=$seed",
            )

            try
                # No persistent state is mutated until generation, both solves,
                # and all exact audits have succeeded.
                sample = generate_sample(sample_id)
                outcomes = solve_both_lps(sample)
                commit_sample!(state, sample, outcomes)

                for outcome in outcomes
                    label = outcome["outcome_kind"] == "affine_equality" ?
                        "affine equality" :
                        (outcome["candidate"]["is_facet"] ? "facet" : "nonfacet")
                    @printf(
                        "  %-10s | %-15s | normalized gap = % .9e | noise = %s\n",
                        outcome["lp_kind"],
                        label,
                        outcome["normalized_gap"],
                        float_string(outcome["critical_noise_fraction"]),
                    )
                end

                write_checkpoint(paths["checkpoint"], state)
                if sample_id % write_tables_every == 0 || sample_id == total_samples
                    write_all_tables!(paths, state, VERTEX_FINGERPRINT)
                end
            catch error_value
                error_value isa InterruptException && rethrow()
                failure = Dict{String, Any}(
                    "timestamp" => string(now()),
                    "sample_id" => sample_id,
                    "ensemble" => ensemble,
                    "stage" => "generate_solve_or_audit",
                    "message" => sprint(showerror, error_value),
                )
                push!(state["failures"], failure)
                append_failure(
                    paths["failures"],
                    sample_id,
                    ensemble,
                    failure["stage"],
                    error_value,
                )
                write_checkpoint(paths["checkpoint"], state)

                if stop_on_error
                    rethrow()
                else
                    @warn "Skipping failed sample" sample_id exception=(error_value, catch_backtrace())
                    state["next_sample_id"] = sample_id + 1
                    write_checkpoint(paths["checkpoint"], state)
                end
            end
        end
    finally
        write_checkpoint(paths["checkpoint"], state)
        write_all_tables!(paths, state, VERTEX_FINGERPRINT)
    end

    elapsed_minutes = (time() - started_at) / 60
    violating_runs = count(run -> run["target_violates"], state["lp_runs"])
    violating_candidates = count(
        candidate -> candidate["best_normalized_gap"] > NORMALIZED_VIOLATION_TOLERANCE,
        state["candidates"],
    )
    facet_candidates = count(candidate -> candidate["is_facet"], state["candidates"])

    println("\nBlind batch complete")
    println("  completed quantum samples: ", length(state["samples"]))
    println("  LP runs:                   ", length(state["lp_runs"]))
    println("  target-violating LP runs:  ", violating_runs)
    println("  distinct inequalities:     ", length(state["candidates"]))
    println("  distinct facets:            ", facet_candidates)
    println("  violating inequalities:     ", violating_candidates)
    @printf("  elapsed this call:          %.2f minutes\n", elapsed_minutes)
    println("  results directory:          ", abspath(output_directory))

    return state
end

run_blind_search! (generic function with 1 method)

## 11. Run or resume the blind batch

This is the only long-running cell. Interrupting it with Jupyter's stop button still triggers the `finally` block, which writes the current checkpoint and tables. Re-running the cell resumes from the next uncommitted sample.

In [30]:
search_state = run_blind_search!()


Sample 1/10000 | haar_general #1 | seed=31001101
  fixed_beta | affine equality | normalized gap =  NaN | noise = 
  free_beta  | facet           | normalized gap = -5.116621294e-03 | noise = 

Sample 2/10000 | haar_general #2 | seed=31001102
  fixed_beta | affine equality | normalized gap =  NaN | noise = 
  free_beta  | facet           | normalized gap = -3.185384852e-03 | noise = 

Sample 3/10000 | haar_general #3 | seed=31001103
  fixed_beta | affine equality | normalized gap =  NaN | noise = 
  free_beta  | facet           | normalized gap = -1.532644391e-02 | noise = 

Sample 4/10000 | haar_general #4 | seed=31001104
  fixed_beta | affine equality | normalized gap =  NaN | noise = 
  free_beta  | facet           | normalized gap = -1.964178332e-02 | noise = 

Sample 5/10000 | haar_general #5 | seed=31001105
  fixed_beta | affine equality | normalized gap =  NaN | noise = 
  free_beta  | facet           | normalized gap = -2.294430517e-02 | noise = 

Sample 6/10000 | haar_general

Excessive output truncated after 524308 bytes.

Dict{String, Any} with 8 entries:
  "format_version"  => 1
  "candidate_index" => Dict("0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…
  "config"          => Dict{String, Any}("vertex_fingerprint"=>"93671de2a20e8f3…
  "next_sample_id"  => 10001
  "samples"         => Any[Dict{String, Any}("sample_id"=>1, "ensemble"=>"haar_…
  "lp_runs"         => Any[Dict{String, Any}("outcome_kind"=>"affine_equality",…
  "failures"        => Any[]
  "candidates"      => Any[Dict{String, Any}("best_critical_visibility"=>NaN, "…

## 12. Inspect the ranking in Julia

The TSV files are the primary portable outputs. This final cell also prints the best distinct candidates found in the current run. A positive normalized gap means the sampled quantum behaviour violates the reconstructed exact inequality. NPA and dedicated see-saw optimization are intentionally deferred until this blind batch is complete.

In [32]:
violating_candidates = filter(
    candidate -> candidate["best_normalized_gap"] > NORMALIZED_VIOLATION_TOLERANCE,
    search_state["candidates"],
)
ranked_candidates = sort(
    violating_candidates;
    by=candidate -> candidate["best_critical_noise_fraction"],
    rev=true,
)

println("\nBest distinct inequalities from this blind batch")
println("rank  facet  face-dim  LP source(s)           sample   normalized gap   white-noise fraction")

for (rank_index, candidate) in enumerate(first(ranked_candidates, min(20, length(ranked_candidates))))
    @printf(
        "%4d  %5s  %8d  %-21s  %6d   % .9e   % .9e\n",
        rank_index,
        string(candidate["is_facet"]),
        candidate["face_dimension"],
        join(candidate["lp_kinds"], ","),
        candidate["best_sample_id"],
        candidate["best_normalized_gap"],
        candidate["best_critical_noise_fraction"],
    )
end

isempty(ranked_candidates) && println("  No target violation was found in the configured batch.")
println("\nPortable ranking file: ", abspath(output_paths(SEARCH_OUTPUT_DIRECTORY)["ranking"]))


Best distinct inequalities from this blind batch
rank  facet  face-dim  LP source(s)           sample   normalized gap   white-noise fraction
  No target violation was found in the configured batch.

Portable ranking file: C:\ResearchRuns\cycle_random_quantum_target_results_general_10000\candidate_ranking.tsv


## Interpretation and next stage

A successful row in `candidate_ranking.tsv` establishes the following, within the recorded numerical quantum evaluation:

- the coefficient vector and bound define an inequality valid on **all** 8,128 corrected cycle vertices;
- the exact face dimension states whether the inequality is a facet;
- the saved state and projectors regenerate the target behaviour by the Born rule;
- `best_critical_noise_fraction` compares candidates in a scale-independent way for their best sampled target.

The appropriate next notebook should take only the most promising distinct candidates, recompute their quantum values at high precision, run NPA upper bounds, and perform dedicated multistart see-saws. Keeping that optimization outside this notebook preserves the blind-discovery logic: the random batch is not steered by recognition of an inequality found partway through it.